# Python Data Analysis Toolkit — Tutorial

This notebook demonstrates the 4 core capabilities of the toolkit:
data cleaning, anomaly detection, report generation, and SQL analysis.

In [8]:
import pandas as pd
import numpy as np
from src import DataCleaner, AnomalyDetector, ReportGenerator, QueryOptimizer

## 1. Sample Data

We create a dataset with missing values and duplicates to clean.

In [9]:
df = pd.DataFrame({
    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Laptop", "Mouse", None],
    "precio": [999.99, 25.50, 75.00, 299.99, 999.99, 25.50, np.nan],
    "stock": [10, 150, 80, 25, 10, 150, 200],
    "categoria": ["Electrónica", "Accesorios", "Accesorios", "Electrónica", "Electrónica", "Accesorios", "Accesorios"]
})

df

,producto,precio,stock,categoria
0,Laptop,999.99,10,Electrónica
1,Mouse,25.50,150,Accesorios
2,Teclado,75.00,80,Accesorios
3,Monitor,299.99,25,Electrónica
4,Laptop,999.99,10,Electrónica
5,Mouse,25.50,150,Accesorios
6,NaN,NaN,200,Accesorios


## 2. Data Cleaning

We impute missing values with `mode`, remove duplicates, and export.

In [10]:
cleaner = DataCleaner(df)
cleaner.handle_missing_values(strategy="mode")
cleaner.remove_duplicates(keep="first")
cleaner.to_csv("output_cleaned.csv")
cleaner.data

2026-09-24 21:43:13 - src.data_cleaner - INFO - handle_missing_values: strategy=mode, columns=None, rows=7


,producto,precio,stock,categoria
0,Laptop,999.99,10,Electrónica
1,Mouse,25.50,150,Accesorios
2,Teclado,75.00,80,Accesorios
3,Monitor,299.99,25,Electrónica
6,Laptop,25.50,200,Accesorios


## 3. Anomaly Detection

We use combined IQR + Z-score to identify outliers.

In [11]:
detector = AnomalyDetector(cleaner.data)
flags = detector.detect(method="both")
flags

,precio_iqr_anomaly,stock_iqr_anomaly,precio_zscore_anomaly,stock_zscore_anomaly,precio_anomaly,stock_anomaly
0,True,False,False,False,True,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
6,False,False,False,False,False,False


## 4. Anomalous Rows

We extract only the rows containing outliers.

In [12]:
detector.get_anomaly_rows()

,producto,precio,stock,categoria
0,Laptop,999.99,10,Electrónica


## 5. Report Generation

We create a Markdown report with descriptive statistics.

In [13]:
generator = ReportGenerator(cleaner.data)
report = generator.generate(format="markdown")
print(report)

# Data Analysis Report

## Overview

- **Rows:** 5
- **Columns:** 4
- **Memory usage:** 0.00 MB
- **Generated (UTC):** 2026-09-25 02:43:14

**Column types:**
- `str`: 2
- `float64`: 1
- `int64`: 1

## Missing Values

No missing values detected.

## Descriptive Statistics

|       |   precio |    stock |
|:------|---------:|---------:|
| count |    5     |   5      |
| mean  |  285.196 |  93      |
| std   |  415.396 |  81.2096 |
| min   |   25.5   |  10      |
| 25%   |   25.5   |  25      |
| 50%   |   75     |  80      |
| 75%   |  299.99  | 150      |
| max   |  999.99  | 200      |

## Sample Data

| producto   |   precio |   stock | categoria   |
|:-----------|---------:|--------:|:------------|
| Laptop     |   999.99 |      10 | Electrónica |
| Mouse      |    25.5  |     150 | Accesorios  |
| Teclado    |    75    |      80 | Accesorios  |
| Monitor    |   299.99 |      25 | Electrónica |
| Laptop     |    25.5  |     200 | Accesorios  |



## 6. Static SQL Analysis

We analyze a query without a database connection.

In [14]:
optimizer = QueryOptimizer()
analysis = optimizer.analyze("SELECT * FROM productos WHERE precio > 100")
print(optimizer.generate_report(analysis, format="text"))

SQL Query Analysis
Complexity: low
Tables: productos
Columns: *

Features:
  SELECT *: True
  WHERE: True
  JOINs: 0
  GROUP BY: False
  ORDER BY: False
  LIMIT: False

Warnings:
  - Query uses SELECT * which may retrieve unnecessary columns.

Suggestions:
  - Specify only required columns in the SELECT clause.

Query:
SELECT * FROM productos WHERE precio > 100
